In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from pyspark.sql import SparkSession, functions as F

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")

PROJECT = find_project_root(Path.cwd())
FIGURES = PROJECT / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"

def save(fig, n, name):
    path = FIGURES / f"fig{n:02d}_{name}.png"
    fig.savefig(path)
    plt.close(fig)
    print(f"  saved {path.name}")

print("Figures ->", FIGURES)

Figures -> E:\BODS-project\docs\figures


In [2]:
spark = (SparkSession.builder
         .appName("ST5011CEM_EDA")
         .master("local[*]")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.adaptive.enabled", "false")
         .config("spark.sql.session.timeZone", "Europe/London")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

delays = spark.read.parquet((PROJECT / "data" / "processed" / "observed_delays").as_posix())
delays = (delays
          .withColumn("hour",  F.hour("recorded_ts"))
          .withColumn("dow",   F.date_format("recorded_ts", "E"))
          .withColumn("dow_n", F.dayofweek("recorded_ts"))
          .withColumn("is_weekend", (F.dayofweek("recorded_ts").isin([1, 7])).cast("int"))
          .withColumn("is_peak",
                      ((F.hour("recorded_ts").between(7, 9)) |
                       (F.hour("recorded_ts").between(16, 18))).cast("int")))
delays.cache()

n = delays.count()
print(f"Delay observations: {n:,}")
print(f"Partitions        : {delays.rdd.getNumPartitions()}")
delays.createOrReplaceTempView("delays")

Delay observations: 1,201,509
Partitions        : 20


## 1. Dataset profile

In [3]:
print("Coverage:")
delays.select(
    F.min("recorded_ts").alias("from"),
    F.max("recorded_ts").alias("to"),
    F.countDistinct("vehicle_ref").alias("vehicles"),
    F.countDistinct("route_short_name").alias("routes"),
    F.countDistinct("stop_id").alias("stops"),
    F.countDistinct("agency_name").alias("operators"),
    F.countDistinct("obs_date").alias("days"),
).show(truncate=False)

print("Delay statistics (minutes):")
delays.select(
    F.round(F.mean("delay_min"), 3).alias("mean"),
    F.round(F.expr("percentile_approx(delay_min, 0.5)"), 3).alias("median"),
    F.round(F.stddev("delay_min"), 3).alias("std_dev"),
    F.round(F.min("delay_min"), 2).alias("min"),
    F.round(F.max("delay_min"), 2).alias("max"),
    F.round(F.skewness("delay_min"), 3).alias("skewness"),
    F.round(F.kurtosis("delay_min"), 3).alias("kurtosis"),
).show(truncate=False)

Coverage:
+-------------------+-------------------+--------+------+-----+---------+----+
|from               |to                 |vehicles|routes|stops|operators|days|
+-------------------+-------------------+--------+------+-----+---------+----+
|2026-07-23 10:09:29|2026-07-27 08:10:54|2020    |357   |14741|24       |5   |
+-------------------+-------------------+--------+------+-----+---------+----+

Delay statistics (minutes):
+-----+------+-------+-----+----+--------+--------+
|mean |median|std_dev|min  |max |skewness|kurtosis|
+-----+------+-------+-----+----+--------+--------+
|0.503|0.35  |5.553  |-30.0|30.0|-0.032  |7.448   |
+-----+------+-------+-----+----+--------+--------+



In [4]:
print("Data quality — null counts across key columns:")
cols = ["delay_min", "hour", "stop_sequence", "dist_m", "agency_name",
        "route_short_name", "direction_id", "stop_lat", "stop_lon"]
delays.select([F.sum(F.col(x).isNull().cast("int")).alias(x) for x in cols]).show(truncate=False)

print("Matching quality — distance from stop at inferred arrival (metres):")
delays.select(
    F.round(F.mean("dist_m"), 1).alias("mean"),
    F.round(F.expr("percentile_approx(dist_m, 0.5)"), 1).alias("median"),
    F.round(F.expr("percentile_approx(dist_m, 0.9)"), 1).alias("p90"),
    F.round(F.max("dist_m"), 1).alias("max"),
).show()

Data quality — null counts across key columns:
+---------+----+-------------+------+-----------+----------------+------------+--------+--------+
|delay_min|hour|stop_sequence|dist_m|agency_name|route_short_name|direction_id|stop_lat|stop_lon|
+---------+----+-------------+------+-----------+----------------+------------+--------+--------+
|0        |0   |0            |0     |0          |0               |0           |0       |0       |
+---------+----+-------------+------+-----------+----------------+------------+--------+--------+

Matching quality — distance from stop at inferred arrival (metres):
+----+------+----+-----+
|mean|median| p90|  max|
+----+------+----+-----+
|32.8|  28.3|70.7|100.0|
+----+------+----+-----+



## 2. Reliability under three definitions

The brief defines urban Service Reliability as +/- 2 minutes and rural as
+/- 5 minutes. The Department for Transport's published bus punctuality
statistics instead use an asymmetric window of 1 minute early to 5 minutes late,
on the basis that early departure inconveniences passengers more than a modest
late arrival.

All three are reported so the headline figure can be stated against the brief's
threshold while acknowledging how it compares to the official measure.

In [5]:
delays = (delays
          .withColumn("on_time_2min",  (F.abs("delay_min") <= 2).cast("int"))
          .withColumn("on_time_5min",  (F.abs("delay_min") <= 5).cast("int"))
          .withColumn("on_time_dft",   ((F.col("delay_min") >= -1) &
                                        (F.col("delay_min") <= 5)).cast("int")))
delays.createOrReplaceTempView("delays")

spark.sql("""
    SELECT COUNT(*)                                     AS observations,
           ROUND(100.0*SUM(on_time_2min)/COUNT(*), 2)   AS pct_within_2min_urban,
           ROUND(100.0*SUM(on_time_5min)/COUNT(*), 2)   AS pct_within_5min_rural,
           ROUND(100.0*SUM(on_time_dft)/COUNT(*), 2)    AS pct_dft_window,
           ROUND(AVG(delay_min), 2)                     AS mean_delay_min
    FROM delays
""").show(truncate=False)

+------------+---------------------+---------------------+--------------+--------------+
|observations|pct_within_2min_urban|pct_within_5min_rural|pct_dft_window|mean_delay_min|
+------------+---------------------+---------------------+--------------+--------------+
|1201509     |54.15                |81.79                |61.91         |0.5           |
+------------+---------------------+---------------------+--------------+--------------+



## 3. Figure 1 — Distribution of delay

In [6]:
# Histogram computed in Spark, not by collecting raw rows
BINS = 60
LO, HI = -20, 20
hist = (delays
        .filter(F.col("delay_min").between(LO, HI))
        .withColumn("bin", F.floor((F.col("delay_min") - LO) / ((HI - LO) / BINS)))
        .groupBy("bin").count()
        .orderBy("bin"))
pdf = hist.toPandas()                      # <- reduced to 60 rows before collecting
pdf["delay_min"] = LO + pdf["bin"] * ((HI - LO) / BINS)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(pdf["delay_min"], pdf["count"], width=(HI-LO)/BINS, color="#3b7dd8", edgecolor="none")
ax.axvline(0, color="black", lw=1, ls="--", label="On schedule")
ax.axvspan(-2, 2, color="green", alpha=0.12, label="Within +/-2 min")
ax.set_xlabel("Delay (minutes)   negative = early, positive = late")
ax.set_ylabel("Observations")
ax.set_title("Distribution of Observed Bus Arrival Delay")
ax.legend()
save(fig, 1, "delay_distribution")

  saved fig01_delay_distribution.png


## 4. Figure 2 — Delay by hour of day

In [7]:
by_hour = spark.sql("""
    SELECT hour,
           COUNT(*)                                   AS observations,
           ROUND(AVG(delay_min), 3)                   AS mean_delay,
           ROUND(100.0*SUM(on_time_2min)/COUNT(*), 2) AS on_time_pct
    FROM delays GROUP BY hour ORDER BY hour
""").toPandas()

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
a1.plot(by_hour["hour"], by_hour["mean_delay"], marker="o", color="#d1495b")
a1.axhline(0, color="grey", lw=1, ls="--")
a1.fill_between(by_hour["hour"], 0, by_hour["mean_delay"], alpha=0.2, color="#d1495b")
a1.set_ylabel("Mean delay (min)")
a1.set_title("Delay and Punctuality by Hour of Day")

a2.bar(by_hour["hour"], by_hour["on_time_pct"], color="#3b7dd8")
a2.axhline(85, color="green", ls="--", lw=1.2, label="85% target (brief)")
a2.set_xlabel("Hour of day")
a2.set_ylabel("Within +/-2 min (%)")
a2.set_xticks(range(0, 24))
a2.legend()
save(fig, 2, "delay_by_hour")
by_hour

  saved fig02_delay_by_hour.png


,hour,observations,mean_delay,on_time_pct
0,0,5,-9.563,20.00
1,1,2,1.208,0.00
2,3,1,8.950,0.00
3,4,228,-5.069,51.75
4,5,4374,-0.724,53.02
5,6,33,0.184,45.45
6,7,31259,-0.175,55.93
7,8,16512,-0.099,56.46
8,9,310,-0.247,56.77
9,10,76502,0.486,54.67


## 5. Figure 3 — Operator compliance

In [8]:
by_op = spark.sql("""
    SELECT agency_name                                 AS operator,
           COUNT(*)                                    AS observations,
           ROUND(100.0*SUM(on_time_2min)/COUNT(*), 2)  AS reliability_pct,
           ROUND(AVG(delay_min), 2)                    AS mean_delay,
           ROUND(STDDEV(delay_min), 2)                 AS delay_sd
    FROM delays GROUP BY agency_name
    HAVING COUNT(*) >= 100
    ORDER BY reliability_pct DESC
""").toPandas()

fig, ax = plt.subplots(figsize=(9, max(4, 0.36*len(by_op))))
colors = ["#2a9d8f" if v >= 85 else "#e9c46a" if v >= 50 else "#e76f51"
          for v in by_op["reliability_pct"]]
ax.barh(by_op["operator"], by_op["reliability_pct"], color=colors)
ax.axvline(85, color="green", ls="--", lw=1.2, label="85% target (brief)")
ax.invert_yaxis()
ax.set_xlabel("Services within +/-2 minutes of timetable (%)")
ax.set_title("Operator Compliance against Service Reliability Threshold")
ax.legend()
save(fig, 3, "operator_compliance")
by_op

  saved fig03_operator_compliance.png


,operator,observations,reliability_pct,mean_delay,delay_sd
0,Howards Travel,225,67.11,-0.36,6.83
1,Stagecoach Cumbria and North Lancashire,16537,60.17,0.12,4.15
2,Bee Network,1023236,55.59,0.46,5.15
3,The Blackburn Bus Company,21867,54.34,0.71,4.50
4,Arriva North West,61592,47.26,0.47,6.36
5,Hattons Travel,680,46.62,2.41,3.31
6,Warrington's Own Buses,34997,45.36,1.04,8.05
7,First Halifax,2093,43.10,1.04,8.74
8,Ashcroft Travel,1950,38.41,2.19,9.98
9,Stagecoach Merseyside and South Lancashire,1535,37.85,0.79,10.99


## 6. Figure 4 — Weekday against weekend

In [9]:
by_dow = spark.sql("""
    SELECT dow, MIN(dow_n) AS ord,
           COUNT(*)                                   AS observations,
           ROUND(AVG(delay_min), 3)                   AS mean_delay,
           ROUND(100.0*SUM(on_time_2min)/COUNT(*), 2) AS on_time_pct
    FROM delays GROUP BY dow ORDER BY ord
""").toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(by_dow["dow"], by_dow["mean_delay"],
              color=["#e76f51" if d in ("Sat", "Sun") else "#3b7dd8" for d in by_dow["dow"]])
ax.axhline(0, color="grey", lw=1)
ax.set_ylabel("Mean delay (minutes)")
ax.set_title("Mean Delay by Day of Week")
for b, v in zip(bars, by_dow["observations"]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f"n={v:,}",
            ha="center", va="bottom", fontsize=7)
save(fig, 4, "delay_by_weekday")
by_dow

  saved fig04_delay_by_weekday.png


,dow,ord,observations,mean_delay,on_time_pct
0,Sun,1,59172,0.541,57.40
1,Mon,2,47652,-0.146,56.12
2,Thu,5,215,1.116,41.86
3,Fri,6,673832,0.644,51.95
4,Sat,7,420638,0.345,56.99


## 7. Figure 5 — Position along the route

In [10]:
by_seq = spark.sql("""
    SELECT stop_sequence, COUNT(*) AS observations,
           ROUND(AVG(delay_min), 3) AS mean_delay
    FROM delays
    WHERE stop_sequence <= 40
    GROUP BY stop_sequence HAVING COUNT(*) >= 20
    ORDER BY stop_sequence
""").toPandas()

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(by_seq["stop_sequence"], by_seq["mean_delay"], marker="o", color="#6a4c93")
ax.axhline(0, color="grey", ls="--", lw=1)
ax.set_xlabel("Stop sequence (position along route)")
ax.set_ylabel("Mean delay (minutes)")
ax.set_title("Delay Accumulation Along the Route")
save(fig, 5, "delay_by_sequence")
by_seq.head(15)

  saved fig05_delay_by_sequence.png


,stop_sequence,observations,mean_delay
0,0,20936,-0.127
1,1,22174,0.569
2,2,22057,0.669
3,3,21066,0.610
4,4,22252,0.594
5,5,22538,0.481
6,6,22829,0.431
7,7,23313,0.336
8,8,23399,0.370
9,9,23737,0.434


## 8. Figure 6 — Correlation between numeric features

In [11]:
num_cols = ["delay_min", "hour", "stop_sequence", "dist_m",
            "is_peak", "is_weekend", "arrival_sec", "stop_lat", "stop_lon"]

# Correlations computed pairwise in Spark; only the 9x9 matrix reaches the driver
corr = pd.DataFrame(index=num_cols, columns=num_cols, dtype=float)
for i, a in enumerate(num_cols):
    for b in num_cols[i:]:
        v = delays.stat.corr(a, b) if a != b else 1.0
        corr.loc[a, b] = corr.loc[b, a] = v

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr.astype(float), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, cbar_kws={"label": "Pearson r"})
ax.set_title("Correlation Matrix of Candidate Features")
save(fig, 6, "correlation_matrix")
corr.round(3)

  saved fig06_correlation_matrix.png


,delay_min,hour,stop_sequence,dist_m,is_peak,is_weekend,arrival_sec,stop_lat,stop_lon
delay_min,1.000,-0.002,0.009,-0.013,-0.032,-0.020,-0.034,0.003,-0.006
hour,-0.002,1.000,0.018,-0.002,0.517,0.280,0.995,-0.003,0.011
stop_sequence,0.009,0.018,1.000,0.045,-0.000,0.002,0.016,0.085,-0.030
dist_m,-0.013,-0.002,0.045,1.000,0.013,0.035,-0.002,0.010,-0.029
is_peak,-0.032,0.517,-0.000,0.013,1.000,0.113,0.518,-0.003,0.012
is_weekend,-0.020,0.280,0.002,0.035,0.113,1.000,0.283,-0.007,0.008
arrival_sec,-0.034,0.995,0.016,-0.002,0.518,0.283,1.000,-0.003,0.011
stop_lat,0.003,-0.003,0.085,0.010,-0.003,-0.007,-0.003,1.000,-0.059
stop_lon,-0.006,0.011,-0.030,-0.029,0.012,0.008,0.011,-0.059,1.000


## 9. Figure 7 — Geographic distribution of delay

In [12]:
by_stop = spark.sql("""
    SELECT stop_lat, stop_lon,
           COUNT(*)                 AS observations,
           ROUND(AVG(delay_min), 3) AS mean_delay
    FROM delays
    GROUP BY stop_lat, stop_lon
    HAVING COUNT(*) >= 10
""").toPandas()

fig, ax = plt.subplots(figsize=(8.5, 7.5))
sc = ax.scatter(by_stop["stop_lon"], by_stop["stop_lat"],
                c=by_stop["mean_delay"].clip(-5, 5),
                s=(by_stop["observations"] ** 0.5).clip(4, 60),
                cmap="RdYlGn_r", alpha=0.75, edgecolors="none")
plt.colorbar(sc, ax=ax, label="Mean delay (minutes)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"Mean Delay by Stop Location  (n={len(by_stop):,} stops)")
save(fig, 7, "geographic_delay")
print(f"Stops plotted: {len(by_stop):,}")

  saved fig07_geographic_delay.png
Stops plotted: 14,003


## 10. Figure 8 — Delay spread by hour

In [13]:
q = spark.sql("""
    SELECT hour,
           ROUND(percentile_approx(delay_min, 0.10), 2) AS p10,
           ROUND(percentile_approx(delay_min, 0.25), 2) AS p25,
           ROUND(percentile_approx(delay_min, 0.50), 2) AS p50,
           ROUND(percentile_approx(delay_min, 0.75), 2) AS p75,
           ROUND(percentile_approx(delay_min, 0.90), 2) AS p90
    FROM delays GROUP BY hour ORDER BY hour
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.fill_between(q["hour"], q["p10"], q["p90"], alpha=0.20, color="#3b7dd8", label="10th-90th pct")
ax.fill_between(q["hour"], q["p25"], q["p75"], alpha=0.40, color="#3b7dd8", label="25th-75th pct")
ax.plot(q["hour"], q["p50"], color="#14213d", marker="o", lw=1.8, label="Median")
ax.axhline(0, color="grey", ls="--", lw=1)
ax.set_xlabel("Hour of day"); ax.set_ylabel("Delay (minutes)")
ax.set_title("Delay Spread by Hour — Travel Time Variability")
ax.set_xticks(range(0, 24)); ax.legend()
save(fig, 8, "delay_spread_by_hour")
q

  saved fig08_delay_spread_by_hour.png


,hour,p10,p25,p50,p75,p90
0,0,-26.85,-17.15,-13.67,0.45,9.40
1,1,-3.10,-3.10,-3.10,5.52,5.52
2,3,8.95,8.95,8.95,8.95,8.95
3,4,-24.03,-9.57,-0.58,0.82,1.98
4,5,-8.10,-1.65,0.32,1.85,4.37
5,6,-4.12,-1.40,0.42,2.43,4.88
6,7,-4.67,-1.80,-0.20,1.62,4.22
7,8,-4.35,-1.75,-0.17,1.58,4.20
8,9,-4.35,-2.10,-0.25,1.25,3.62
9,10,-3.97,-1.17,0.35,2.22,5.12


## 11. Summary

All figures are in `docs/figures/`. Each should appear in the report numbered and
captioned, per the template.

In [14]:
figs = sorted(FIGURES.glob("*.png"))
print(f"{len(figs)} figures generated:\n")
for f in figs:
    print(f"  {f.name:<38} {f.stat().st_size/1024:7.1f} KB")

14 figures generated:

  fig01_delay_distribution.png              28.7 KB
  fig02_delay_by_hour.png                   45.9 KB
  fig03_operator_compliance.png             72.9 KB
  fig04_delay_by_weekday.png                20.8 KB
  fig05_delay_by_sequence.png               40.9 KB
  fig06_correlation_matrix.png              80.8 KB
  fig07_geographic_delay.png               320.7 KB
  fig08_delay_spread_by_hour.png            63.6 KB
  fig09_model_comparison.png                47.6 KB
  fig10_feature_importance.png              43.1 KB
  fig11_predicted_vs_actual.png             71.7 KB
  fig12_error_by_operator.png               49.4 KB
  fig13_horizon_comparison.png              40.7 KB
  fig14_schema_diagram.png                  66.7 KB


In [15]:
spark.stop()
print("Notebook 03 complete.")

Notebook 03 complete.


In [15]:
print("driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

driver memory: 4g
